In [1]:
%pip install langgraph


   -------- ------------------------------- 1/5 [langgraph-sdk]
   ---------------- ----------------------- 2/5 [langgraph-checkpoint]
   ---------------- ----------------------- 2/5 [langgraph-checkpoint]
   ------------------------ --------------- 3/5 [langgraph-prebuilt]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   -------------------------------- ------- 4/5 [langgraph]
   ---------------------------------------- 5/5 [langgraph]

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_openai import ChatOpenAI

# 모델 초기화
model=ChatOpenAI(model="gpt-4o-mini")
model.invoke('안녕하세요!')

AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 10, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CHrnWWuQZtp4juho9zmXy9SX1eGZF', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0cad58ae-7e51-44c5-a596-b3e496e7131e-0', usage_metadata={'input_tokens': 10, 'output_tokens': 11, 'total_tokens': 21, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [3]:
from typing import Annotated    # annotated는 타입 힌트를 사용할 때 사용하는 점수
from typing_extensions import TypedDict # TypedDict는 딕셔너리 타입을 정의할 때 사용하는 함수

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    """
    State 클래스는 TypedDict를 상속받습니다.

    속성:
        messages(Annoted[list[str], add_messages]): 메시지들은 "list" 타입을 가집니다.
        'add_messages' 함수는 이 상태 키가 어떻게 업데이트되어야 하는지를 정의합니다.
    """
    messages: Annotated[list[str], add_messages]

# StateGraph 클래스를 사용하여 State 타입의 그래프 생성
graph_builder=StateGraph(State)

In [4]:
def generate(state:State):
    """
    주어진 상태를 기반으로 챗봇의 응답 메시지를 생성합니다.

    매개변수:
    state (State): 현재 대화 상태를 나타내는 객체로, 이전 메시지들이 포함되어 있습니다.

    반환값:
    dict: 모델이 생성한 응답 메시지를 포함하는 딕셔너리.
        형식은 {"messages": [응답 메시지]}입니다.
    """

    return {"messages": [model.invoke(state["messages"])]}

graph_builder.add_node("generate", generate)

In [8]:
graph_builder.add_edge(START,"generate")
graph_builder.add_edge("generate", END)

graph=graph_builder.compile()

In [6]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass

In [9]:
response = graph.invoke({"messages":["안녕하세요! 저는 오성수입니다."]})

print(type(response))
response

<class 'dict'>


{'messages': [HumanMessage(content='안녕하세요! 저는 오성수입니다.', additional_kwargs={}, response_metadata={}, id='462f404e-d1a9-4c3a-be42-7e73ba1a2d49'),
  AIMessage(content='안녕하세요, 오성수님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 16, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CHro8mEBxs86PUGMlKJSR0Ah4KceD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--400c6d6f-c627-418d-a993-4ae1af42da48-0', usage_metadata={'input_tokens': 16, 'output_tokens': 21, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

In [10]:
response["messages"].append("제 이름을 아시나요?")
graph.invoke(response)

{'messages': [HumanMessage(content='안녕하세요! 저는 오성수입니다.', additional_kwargs={}, response_metadata={}, id='462f404e-d1a9-4c3a-be42-7e73ba1a2d49'),
  AIMessage(content='안녕하세요, 오성수님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 16, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CHro8mEBxs86PUGMlKJSR0Ah4KceD', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--400c6d6f-c627-418d-a993-4ae1af42da48-0', usage_metadata={'input_tokens': 16, 'output_tokens': 21, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
  HumanMessage(con

In [11]:
inputs={"messages":[("human","한국과 일본의 관계에 대해 자세히 알려 줘.")]}
for chunk, _ in graph.stream(inputs, stream_mode="messages"):
    print(chunk.content, end="")

한국과 일본의 관계는 역사적으로 복잡하고 다층적입니다. 두 나라는 지리적으로 가까운 이웃국가지만, 역사적 사건과 정치적 이견으로 인해 갈등이 빈발해왔습니다. 다음은 한국과 일본의 관계를 이해하는 데 필요한 주요 요소들입니다.

### 역사적 배경

1. **일제 강점기 (1910-1945)**:
   - 일본은 1910년부터 1945년까지 한국을 식민지로 지배했습니다. 이 시기에 한국인들은 많은 인권 침해와 문화적 억압을 경험했습니다. 이 시절의 경험은 현재까지도 한국 사회에서 깊은 상처로 남아 있습니다.

2. **전후 재조정**:
   - 제2차 세계대전 후, 한국은 1945년에 해방되었고, 일본은 패전국으로서 전후 처리를 받았습니다. 1965년, 한국과 일본은 "한일기본조약"을 체결하여 외교 관계를 정상화하고, 일본이 한국에 대해 재정 지원을 하기로 했습니다. 하지만 식민지 시기의 문제는 여전히 해결되지 않았습니다.

### 현재의 갈등 요소

1. **역사 문제**:
   - 일본의 식민지 지배 기간 동안의 역사 문제, 특히 강제징용, 위안부 문제 등이 여전히 중요한 갈등 요소입니다. 한국은 일본 정부에 대한 사과와 보상을 요구해왔으며, 일본은 이를 공식적으로 해결했다고 주장하고 있습니다.

2. **영토 분쟁**:
   - 독도(일본명: 다케시마)를 둘러싼 영토 분쟁은 한국과 일본 간의 또 다른 갈등의 뿌리입니다. 두 나라는 각각 독도가 자국의 고유 영토라고 주장하고 있으며, 이 문제는 양국 간의 외교 관계에 부정적인 영향을 미치고 있습니다.

3. **문화적 갈등**:
   - 두 나라 간의 문화적 교류가 있지만, 동시에 서로에 대한 편견과 갈등도 존재합니다. 특히, 일본의 2차 대전 관련 역사 수정주의는 한국 내에서 반발을 일으키고 있습니다.

### 협력의 기회

1. **경제적 연결**:
   - 한국과 일본은 서로 중요한 경제 파트너국입니다. 두 나라는 무역, 기술 협력 및 투자에서 긴밀한 관계를 유지하고 있습니다.

2. **안보 협력**: